# GEMs Project 06/01 - 06/05 Notebook

Continuing work from last week's notebook: `anaerobic_conditions_052726.ipynb`. 

## Objectives
My goal for this week is to:
1. Simulate anaerobic conditions in all the models, 
2. Implement the micronutrient conditions SW sent me, 
3. Create a "demo" notebook. 

## Generate all the Models - 06/01/26

The code to generate models: STM_V1_0, stm1090, stm1585, and stm2575 is copied from last week's notebook. See `/home/gmvaz/2026_GEMs/stm_model_test/anaerobic_conditions_052726.ipynb` for more details. 

In [1]:
import os
from cobra.core.configuration import Configuration

# Set the Gurobi license and solver
os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

In [2]:
from cobra.io import load_model, load_json_model

# load the test S.Tm LT2 GEM in cobrapy
base_model = load_model("STM_v1_0")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu


In [3]:
import pandas as pd

# read in the gene weights csv files of the different threshold rna-seq data
stm_threshold_10_90 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_25_75.csv")

In [4]:
# make each list of gene weights into a python set
stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

# check the length
print(len(stm_set_10_90))
print(len(stm_set_15_85))
print(len(stm_set_25_75))

4671
4671
4671


In [5]:
# make a list of all genes in the base model and save it as a set
model_genes = set([g.id for g in base_model.genes])

#check
print(len(model_genes))

1271


In [6]:
# subset the gene weights list for each model/sample to only have the genes present in the input model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

In [7]:
# there are different amounts of genes in the gene weights lists and in the input model gene list. 
# make a new gene weight list per sample that has all the genes from the input model:
# (1) assign a value of 0 to the 20 genes found in the input model but not in the gene weights list. 
# (2) add the genes that are present in the input model and raw gene weight list. 
# Result: a gene weight list per sample that has only the genes found in the input model and raw gene weights list.

subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

# check
print(len(subset_weights_stm1090))
print(len(subset_weights_stm1585))
print(len(subset_weights_stm2575))

1271
1271
1271


In [8]:
from imatpy.parse_gpr import gene_to_rxn_weights

# convert gene weights to reaction weights using the final gene weights lists with all genes from the input model
rxn_weights_stm1090 = gene_to_rxn_weights(base_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(base_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(base_model, subset_weights_stm2575)

# check
print(len(rxn_weights_stm1090))
print(len(rxn_weights_stm1585))
print(len(rxn_weights_stm2575))

2545
2545
2545


In [9]:
from imatpy.model_creation import generate_model

# generate context-specific models with imat by integrating the reaction weights
stm1090_model = generate_model(model = base_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = base_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = base_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

Read LP format model from file /tmp/tmp35tuwusj.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpu64dj6z6.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpik5yoesq.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


In [10]:
# Check the Models
# WHY ARE ALL THE REACTIONS THE SAMEEEEEEEEEEEEEEEEEEE. WHAT THE HELL IS GOING ON.
# jk I just remembered reactions are turned off, not removed :)
# I should check how many reactions have zero flux though.... that's what I actually want to know. 

print(len(stm1090_model.reactions))
print(len(stm1585_model.reactions))
print(len(stm2575_model.reactions))

2545
2545
2545


In [11]:
# copy the models to run FBA 
base_model_fba = load_model("STM_v1_0")
stm1090_model_fba = stm1090_model.copy()
stm1585_model_fba = stm1585_model.copy()
stm2575_model_fba = stm2575_model.copy()

# run FBA on all models
base_model_fba_res = base_model_fba.optimize()
stm1090_fba_res = stm1090_model_fba.optimize()
stm1585_fba_res = stm1585_model_fba.optimize()
stm2575_fba_res = stm2575_model_fba.optimize()

Read LP format model from file /tmp/tmpz4v_q4cl.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpvuvuaeb7.lp
Reading time = 0.02 seconds
: 3201 rows, 6460 columns, 25176 nonzeros
Read LP format model from file /tmp/tmpy1fx6kg7.lp
Reading time = 0.02 seconds
: 3949 rows, 7128 columns, 28088 nonzeros


In [12]:
# print the objective value for each model after FBA

print(base_model_fba_res.objective_value)
print(stm1090_fba_res.objective_value)
print(stm1585_fba_res.objective_value)
print(stm2575_fba_res.objective_value)

0.4778336607607207
0.4072058061353676
0.40650099245760307
0.3934219732489797


## Simulate an Anaerobic Environment - 06/02/26

The growth media of a model is managed by the `medium` attribute. `model.medium` returns a dictionary that has the upper bounds of all the non-zero exchange fluxes. 

Below is a list of all the active exchange reactions. Note, the list we see below only shows the upper bounds of active exchange reactions, that is to say, it's the maximum import flux of that reaction but that does not gurantee that the organism will consume that metabolite or product.

We see that aerobic growth is enabled because the upper exchange flux is 18.5 `mmol / [gDW h]` (concentration per gram dry weight of cells and hour). 

**MORE INFO ON:**
- **how to read the concentration values**
- **how to decide the concentration values for other metabolites**

In [20]:
base_model_fba.medium

{'EX_ca2_e': 1000.0,
 'EX_cbl1_e': 0.01,
 'EX_cl_e': 1000.0,
 'EX_co2_e': 1000.0,
 'EX_cobalt2_e': 1000.0,
 'EX_cu2_e': 1000.0,
 'EX_mg2_e': 1000.0,
 'EX_fe2_e': 1000.0,
 'EX_fe3_e': 1000.0,
 'EX_mn2_e': 1000.0,
 'EX_mobd_e': 1000.0,
 'EX_na1_e': 1000.0,
 'EX_nh4_e': 1000.0,
 'EX_o2_e': 18.5,
 'EX_pi_e': 1000.0,
 'EX_zn2_e': 1000.0,
 'EX_so4_e': 1000.0,
 'EX_glc__D_e': 5.0,
 'EX_tungs_e': 1000.0,
 'EX_h2o_e': 100.0,
 'EX_h_e': 100.0,
 'EX_k_e': 1000.0}

Let's create copies of each model, change the conditions to be anaerobic, and simulate growth. Each constraint change will be tested on a copy of the "original" models to keep results clean and seperate from each other. 

In [14]:
# Create copies of the original models to test anaerobic condition

no_oxygen_base = load_model("STM_v1_0")

#no_oxygen_stm1585 = stm1585_model.copy()
#no_oxygen_stm2575 = stm2575_model.copy()

Let's make a copy of the `medium` list and remove oxygen from the new copy labeled `no_oxygen_medium`. Then, let's assign `no_oxygen_medium` list to the copy of the base model labeled `no_oxygen_base`.  

In [15]:
# Copy the medium attribute from the base model. Remove oxygen flux. Save as new.

no_oxygen_medium = base_model_fba.medium
no_oxygen_medium["EX_o2_e"] = 0.0
no_oxygen_base.medium = no_oxygen_medium

### Compare the growth rate of the S. Tm `base_model` and anaerobic `base_model`

We now have a copy of `base_model` designated for simulating anaerobic conditions and the anaerobic conditions list applied. Let's simulate FBA and compare growth of the `no_oxygen_base` model with the growth from the `base_model`. 

In [16]:
# Run FBA on the `no_oxygen_base` model. 
no_oxygen_base_fba_res = no_oxygen_base.optimize()

# print the objective value for the anaerobic base_model - control
print(no_oxygen_base_fba_res.objective_value)

# print the objective value for the aerobic base_model - control
print(base_model_fba_res.objective_value)

0.05251095705777986
0.4778336607607207


### Comparing the growth rate of the `stm1090_model` and the anaerobic `stm1090_model`

Let's repeat these steps for the `stm1090_model`. 

1. Create a new medium named `no_oxygen_medium_stm1090`
2. Assign the medium from `stm1090_model` to the new, empty list.
3. Change the oxygen upper bound to 0 in the new list.
4. Assign the new list, `no_oxygen_medium_stm1090`, to the model `no_oxygen_stm_1090`

In [17]:
# create a copy of `stm1090_model` and label it `no_oxygen_stm1090`
no_oxygen_stm1090 = stm1090_model.copy()

# make a new medium dictionary for stm1090 and copy the `no_oxygen_medium_stm1090` medium into it. 
no_oxygen_medium_stm1090 = no_oxygen_stm1090.medium
# WRITE A CHECK FOR THIS

# change the upper bounds of the Oxygen reaction to 0 in `no_oxygen_medium_stm1090`.
no_oxygen_medium_stm1090["EX_o2_e"] = 0.0
# WRITE A CHECK FOR THIS

# assign the anaerobic medium to the `no_oxygen_stm1090` model
no_oxygen_stm1090.medium = no_oxygen_medium_stm1090

Read LP format model from file /tmp/tmpocx31r_l.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros


Let's run FBA on `no_oxygen_medium_stm1090` and print the objective value. 

In [18]:
no_oxygen_stm1090_fba_res = no_oxygen_stm1090.optimize()

print(no_oxygen_stm1090_fba_res.objective_value)

/home/gmvaz/Installs/miniforge3/envs/gems/lib/python3.11/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


GurobiError: Unable to retrieve attribute 'X'

## Results Review - 06/02/26
### GurobiError when running FBA on `no_oxygen_stm1090` model

```
GurobiError: Unable to retrieve attribute `X`.
```
I considered: 
1. Maybe I can't assign the same `medium` attribute to different models.

I initially wrote the code like this:

```
no_oxygen_medium = base_model_fba.medium

no_oxygen_stm1090.medium = no_oxygen_medium
no_oxygen_stm1585.medium = no_oxygen_medium
no_oxygen_stm2575.medium = no_oxygen_medium

no_oxygen_stm1090_fba_res = no_oxygen_stm1090.optimize()
no_oxygen_stm1585_fba_res = no_oxygen_stm1585.optimize()
no_oxygen_stm2575_fba_res = no_oxygen_stm2575.optimize()

print(no_oxygen_stm1090_fba_res.objective_value)
print(no_oxygen_stm1585_fba_res.objective_value)
print(no_oxygen_stm2575_fba_res.objective_value)

```
So, I thought that maybe the issue is that I can't use the same anaerobic medium. 

2. Cellular growth can't be simulated in an anaerobic environment in the reduced models.

I looked at the error more closely and noticed the top line states `solver status infeasible`. This lead me to think that the solution is the attribute Gurobi can't retrieve. 

```
UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
```

## Questions, Thoughts, Notes - 06/02/26
I have questions about this though... Why is it that FBA finds a solution for the control S. Tm model but not for any of the iMAT constrained models? I would be more understanding if it was unable to solve the 25/50/25 threshold model but I'm surprised that the 10/80/10 model can't be solved considering it's the least constrained. Obviously, I know it's still possible I just didn't expect it. 

I wonder if the medium I have is feasible for Salmonella to grow in anaerobic conditions?? Just because I removed oxygen doesn't mean S. Tm can grow in whatever that list says.  I think I should email this to Sebastian to get his eyes on it, it would be quicker than me looking a bunch of stuff up, getting side tracked and ending up confused. Also - I could look into his metabolite list and ask how to apply the concentrations! 

## Implementing Media Conditions sent by SW - 06/03/26

**More Info:**
- **Add a screenshot of the list SW sent**

## Meeting with Titus 06/03/26

Questions: 

- Help with my error.
- How do I check for all the zero flux reactions in the model?
- Should I look into this: how to get the minimal media, `.slim_optimize()` vs `.optimize()`, boundary reactions? See `anaerobic_conditions_052726.ipynb` and `05.29.26` Obsidian notes. 
- How to embed image into a Markdown cell that's saved on my computer?
- How to add a toc to Notebook?